# Building Simple Tokenizers

In the previous notebook, we learned how to split text into tokens and build a vocabulary. Now we'll wrap that logic into reusable tokenizer classes.

**What we'll build:**
- SimpleTokenizerV1: Basic encode/decode, fails on unknown words
- SimpleTokenizerV2: Handles unknown words with a special token

## Setup

First, let's load our text and build the vocabulary (same as notebook 01).

In [1]:
import os
import re
import requests

# Download if needed
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    response = requests.get(url, timeout=30)
    with open("the-verdict.txt", "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Tokenize and build vocabulary
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_words = sorted(set(preprocessed))
vocab = {token: idx for idx, token in enumerate(all_words)}

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 1130


## SimpleTokenizerV1

Our first tokenizer has two methods:
- `encode(text)`: Convert text to a list of token IDs
- `decode(ids)`: Convert IDs back to text

The decode method needs to handle spacing around punctuation. We don't want "Hello , world ." - we want "Hello, world."

In [2]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: token for token, idx in vocab.items()}
    
    def encode(self, text):
        # Split text into tokens
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        
        # Convert to IDs
        ids = [self.str_to_int[token] for token in tokens]
        return ids
    
    def decode(self, ids):
        # Convert IDs back to tokens
        text = " ".join([self.int_to_str[idx] for idx in ids])
        
        # Fix spacing around punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [3]:
tokenizer = SimpleTokenizerV1(vocab)

text = '"""It\'s the last he painted, you know,"\n           Mrs. Gisburn said with pardonable pride."""'
ids = tokenizer.encode(text)
print(f"Token IDs: {ids}")

Token IDs: [1, 1, 1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7, 1, 1, 1]


In [4]:
# Decode back to text
decoded = tokenizer.decode(ids)
print(f"Decoded: {decoded}")

Decoded: """ It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride."""


## The Problem with Unknown Words

V1 breaks if we try to encode a word that wasn't in our training text.

In [5]:
# This will fail - "Hello" isn't in our vocabulary
try:
    tokenizer.encode("Hello, do you like tea?")
except KeyError as e:
    print(f"KeyError: {e}")
    print("The word isn't in our vocabulary!")

KeyError: 'Hello'
The word isn't in our vocabulary!


## SimpleTokenizerV2: Handling Unknown Words

We'll add two special tokens to our vocabulary:
- `<|endoftext|>`: Marks the end of a document
- `<|unk|>`: Represents any word we haven't seen before

In [6]:
# Rebuild vocabulary with special tokens
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: idx for idx, token in enumerate(all_tokens)}

print(f"New vocabulary size: {len(vocab)}")
print(f"\nLast 5 entries:")
for item in list(vocab.items())[-5:]:
    print(item)

New vocabulary size: 1132

Last 5 entries:
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [7]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx: token for token, idx in vocab.items()}
    
    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        
        # Replace unknown tokens with <|unk|>
        tokens = [
            token if token in self.str_to_int else "<|unk|>"
            for token in tokens
        ]
        
        ids = [self.str_to_int[token] for token in tokens]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[idx] for idx in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [8]:
tokenizer = SimpleTokenizerV2(vocab)

# Concatenate two sentences with the end-of-text token
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

print(f"Input text:\n{text}")

Input text:
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [9]:
ids = tokenizer.encode(text)
print(f"\nToken IDs:\n{ids}")


Token IDs:
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [10]:
decoded = tokenizer.decode(ids)
print(f"\nDecoded:\n{decoded}")


Decoded:
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


Notice how "Hello" and "palace" became `<|unk|>` because they weren't in our training vocabulary.

## Summary

We built two tokenizer versions:
- V1: Simple but breaks on unknown words
- V2: Handles unknown words with `<|unk|>` token

**The problem:** Our vocabulary only knows words from the training text. Real LLMs need to handle any input.

**The solution:** Byte Pair Encoding (BPE), which we'll explore next.